# AA 279B HW 4 Prob 1

## Imports

In [2]:
%load_ext autoreload
%autoreload 2

## MATLAB

In [ ]:
import matlab.engine
from matlab.engine import MatlabEngine

import numpy as np
from numpy.linalg import norm
from scipy.integrate import solve_ivp
from scipy.optimize import fsolve
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt
from lamberthub import vallado2013
import plotly.graph_objects as go
from tqdm import tqdm

import os
import sys

sys.path.insert(0, os.path.abspath('..'))

In [ ]:
from aa279b.aa279b_tools import DEG_TO_RAD, AU_TO_KM, rv_from_coe, MU_SUN

In [7]:
# 1. Start the MATLAB process in the background
print("Starting MATLAB engine...")
eng: MatlabEngine = matlab.engine.start_matlab()

Starting MATLAB engine...


In [ ]:
stop = False
if stop:
    eng.quit()

In [10]:
# # 2. Call standard built-in functions
# result_sqrt = eng.sqrt(16.0)
# print(f"Square root of 16: {result_sqrt}")  # Output: 4.0

# # 3. Handle multiple output arguments using 'nargout'
# # By default, nargout=1. To get more outputs, you must explicitly declare it:
# gcd_val, im, jm = eng.gcd(100.0, 80.0, nargout=3)
# print(f"GCD results -> Value: {gcd_val}, Coefficients: {im}, {jm}")

# # 4. Use the MATLAB Workspace dictionary
# # This pushes variables from Python directly into MATLAB's active memory
# eng.workspace['my_py_var'] = 25.0
# output = eng.eval('sqrt(my_py_var)')
# print(f"Workspace evaluation: {output}")  # Output: 5.0



Square root of 16: 4.0
GCD results -> Value: 20.0, Coefficients: 1.0, -1.0
Workspace evaluation: 5.0


## Setup

In [ ]:
def hci_rv(planet_num: int, jd: float):
    """Heliocentric position and velocity using https://ssd.jpl.nasa.gov/planets/approx_pos.html

    Parameters
    ----------
    planet_num : int
        1 (Mercury) to 9 (Pluto)
    jd : float
        Julian Date
        
    Returns
    ----------
    r_hci : np.ndarray (3,)
        Heliocentric position
    v_hci : np.ndarray (3,)
        Heliocentric velocity
    """
    
    assert planet_num >= 1
    assert planet_num <= 9

    tt_jd = (jd - 2451545.0) / 36525.0
    
    elements, rates = eng.AA279j2000_planetary_elements(planet_num, nargout=2)
    
    
    #  1. Value of six elements (angles in degrees)
    a_au, e, i_deg, L_deg, w_bar_deg, raan_deg = [
        x + dx*tt_jd for x,dx in zip(elements[0], rates[0])
    ]
    
    #  2. aop and M
    aop = (w_bar_deg - raan_deg) * DEG_TO_RAD
    M = (L_deg - w_bar_deg) * DEG_TO_RAD
    
    #  Just solve (from prev homework)
    r_hci, v_hci = rv_from_coe(
        a = a_au * AU_TO_KM,
        e = e,
        i = i_deg * DEG_TO_RAD,
        raan = raan_deg * DEG_TO_RAD,
        aop = aop,
        M = M,
        mu = MU_SUN
    )
    
    return r_hci, v_hci

hci_rv(1,1)

    # #  3. Kepler's eq
    # M = wrap_to_pi(M)
    # E = fsolve(lambda E: E - e*np.sin(E) - M, M)[0]
    
    # #  4. Heliocentric coordinates
    # a = a_au*AU_TO_KM
    # x_p = a*(np.cos(E) - e)
    # y_p = a*np.sqrt(1 - e*e)*np.sin(E)
    # z_p = 0
    
    # #  5. r_hci coordinates
    # raan = raan_deg * DEG_TO_RAD
    # i = i_deg * DEG_TO_RAD
    # M = np.array([
    #     [np.cos(aop)*np.cos(raan) - np.sin(aop)*np.sin(raan)*np.cos(i),
    #      -np.sin(aop)*np.cos(raan) - np.cos(aop)*np.sin(raan)*np.cos(i),
    #      0],
    #     [np.cos(aop)*np.sin(raan) + np.sin(aop)*np.cos(raan)*np.cos(i),
    #      -np.sin(aop)*np.sin(raan) + np.cos(aop)*np.cos(raan)*np.cos(i),
    #      0],
    #     [np.sin(aop)*np.sin(i), np.cos(aop)*np.sin(i), 0]
    # ])
    
    # r_hci = M @ np.array([x_p, y_p, z_p])
    
    
    # #  Getting v
    # v_norm = np.sqrt(MU_SUN * (2/norm(r_hci) - 1/a))
    
    # print(a)
    # print(e)
    # print(i)
    
    # a0, e0, i0, l0, w_bar0, Omega0 = elements[0]
    # da0, de0, di0, dl0, dw_bar0, dOmega0 = rates[0]
    
    # # 1. 

# def kepler(M, e):
#     """Vallado 4e eq. 2-4 p. 45 (pdf p. 72)"""
#     E = fsolve(lambda E: E - e*np.sin(E) - M, M)[0]
#     return E
    
    # aop = 
    


(array([ 2.21379162e+08,  6.56862071e+09, -1.66877762e+09]),
 array([-3.9771446 , -0.81628247, -0.44941768]))